In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

# Load the dataset
df = pd.read_csv('Data-raw/Fertilizer.csv')

# Data preprocessing
# Drop the first column (index column)
df = df.drop(df.columns[0], axis=1)

# Check for missing values
print("Missing values:\n", df.isnull().sum())

# Encode the crop names
label_encoder = LabelEncoder()
df['Crop_Encoded'] = label_encoder.fit_transform(df['Crop'])

# Define features and target
X = df[['N', 'P', 'K', 'pH']]
y = df['Crop_Encoded']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# Evaluate the model
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Save the model and encoders
with open('fertilizer_model.pkl', 'wb') as f:
    pickle.dump(model, f)
    
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
    
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model training complete and saved!")

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

In [8]:

# Load the dataset
df = pd.read_csv(r"C:\Users\jayra\OneDrive\Desktop\Harvestify\Data-raw\Fertilizer.csv")


In [9]:
# Data preprocessing
# Drop the first column (index column)
df = df.drop(df.columns[0], axis=1)

In [10]:
# Check for missing values
print("Missing values:\n", df.isnull().sum())

Missing values:
 Crop    0
N       0
P       0
K       0
pH      0
dtype: int64


In [11]:
# Encode the crop names
label_encoder = LabelEncoder()
df['Crop_Encoded'] = label_encoder.fit_transform(df['Crop'])

In [12]:
# Define features and target
X = df[['N', 'P', 'K', 'pH']]
y = df['Crop_Encoded']

In [13]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [15]:
# Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)


,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [16]:
# Evaluate the model
y_pred = model.predict(X_test_scaled)
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")


Model Accuracy: 0.82


In [20]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
import pickle
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv(r"C:\Users\jayra\OneDrive\Desktop\Harvestify\Data-raw\Fertilizer.csv")


# Data preprocessing
# Drop the first column (index column)
df = df.drop(df.columns[0], axis=1)

# Check for duplicates and remove them
print(f"Original dataset shape: {df.shape}")
df = df.drop_duplicates()
print(f"After removing duplicates: {df.shape}")

# Check class distribution
crop_counts = df['Crop'].value_counts()
print("Crop distribution:")
print(crop_counts)

# Encode the crop names
label_encoder = LabelEncoder()
df['Crop_Encoded'] = label_encoder.fit_transform(df['Crop'])

# Define features and target
X = df[['N', 'P', 'K', 'pH']]
y = df['Crop_Encoded']

# Check for class imbalance
print(f"Class distribution: {np.bincount(y)}")

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_scaled, y_train)

print(f"After SMOTE - X_train: {X_train_balanced.shape}, y_train: {y_train_balanced.shape}")

# Try different models with hyperparameter tuning
models = {
    'RandomForest': RandomForestClassifier(random_state=42),
    'GradientBoosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

# Hyperparameter grids for tuning
param_grids = {
    'RandomForest': {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    },
    'GradientBoosting': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [3, 5]
    },
    'SVM': {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto'],
        'kernel': ['rbf', 'linear']
    }
}

best_model = None
best_score = 0
best_model_name = ""

# Train and evaluate models with cross-validation
for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use GridSearchCV for hyperparameter tuning
    grid_search = GridSearchCV(model, param_grids[name], cv=5, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train_balanced, y_train_balanced)
    
    # Get the best model
    model = grid_search.best_estimator_
    
    # Cross-validation score
    cv_scores = cross_val_score(model, X_train_balanced, y_train_balanced, cv=5)
    print(f"{name} CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
    print(f"Best parameters: {grid_search.best_params_}")
    
    # Test score
    test_score = model.score(X_test_scaled, y_test)
    print(f"{name} Test Accuracy: {test_score:.4f}")
    
    if test_score > best_score:
        best_score = test_score
        best_model = model
        best_model_name = name

print(f"\nBest model: {best_model_name} with accuracy: {best_score:.4f}")

# Train the best model on the full training data
best_model.fit(X_train_balanced, y_train_balanced)

# Final evaluation
y_pred = best_model.predict(X_test_scaled)
final_accuracy = accuracy_score(y_test, y_pred)
print(f"\nFinal Model Accuracy: {final_accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

# Save the model and encoders
''' with open('fertilizer_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
    
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
    
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model training complete and saved!")'''

Original dataset shape: (1843, 5)
After removing duplicates: (1843, 5)
Crop distribution:
Crop
Rice                19
Jowar(Sorghum)      19
Barley(JAV)         19
Maize               19
Ragi( naachnnii)    19
                    ..
Lemon Grass         19
Cotton              19
Jute                19
Coffee              19
Sunflower           19
Name: count, Length: 97, dtype: int64
Class distribution: [19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19
 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19
 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19
 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19 19
 19]
After SMOTE - X_train: (1552, 4), y_train: (1552,)

Training RandomForest...
RandomForest CV Accuracy: 0.8492 (+/- 0.0231)
Best parameters: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
RandomForest Test Accuracy: 0.8238

Training GradientBoosting...
GradientBoosting CV Accuracy:

' with open(\'fertilizer_model.pkl\', \'wb\') as f:\n    pickle.dump(best_model, f)\n    \nwith open(\'label_encoder.pkl\', \'wb\') as f:\n    pickle.dump(label_encoder, f)\n    \nwith open(\'scaler.pkl\', \'wb\') as f:\n    pickle.dump(scaler, f)\n\nprint("Model training complete and saved!")'

In [21]:
print(f"\nBest model: {best_model_name} with accuracy: {best_score:.4f}")


Best model: SVM with accuracy: 0.8591


In [18]:
!pip install imbalanced-learn

In [23]:
with open('fertilizer_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
    
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)
    
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model training complete and saved!")

Model training complete and saved!
